In [48]:
from pynq import Overlay, MMIO
import pynq.lib as lib
from pynq import allocate
import numpy as np

ol = Overlay("/root/jupyter_notebooks/rhd_spi_0807/test1/rhd_spi_ip_0806_wrapper.bit")

In [2]:
ol?

In [4]:
# 名稱以你的 block design 為準；在 Vivado Address Editor 可以看到
spi_ip = ol.axi_rhd_spi_master_0   # 假設 instance 名稱為 “…_0”

# 也可列出所有暫存器 (32-bit words)
spi_ip.register_map

AttributeError: register_map only available if the .hwh is provided

In [7]:
print("Has description:", hasattr(ol, "description"))  # 應該要 True

Has description: False


In [49]:
import os, pathlib
bitfile = "/root/jupyter_notebooks/rhd_spi_0807/test1/rhd_spi_ip_0806_wrapper.bit" 
print("bit exists:", os.path.exists(bitfile))
print("hwh exists:", os.path.exists(bitfile.replace(".bit", ".hwh")))
print("exact hwh name:", pathlib.Path(bitfile).with_suffix(".hwh"))

bit exists: True
hwh exists: True
exact hwh name: /root/jupyter_notebooks/rhd_spi_0807/test1/rhd_spi_ip_0806_wrapper.hwh


In [11]:
print(hasattr(ol, "description"))          # 必須 True
print(list(ol.ip_dict.keys()))             # 會顯示 axi_rhd_spi_master_0 ...
spi_ip = ol.axi_rhd_spi_master_0           # 真正屬性名用 ip_dict 中的 key
print(hasattr(spi_ip, "register_map"))     # 也必須 True
dir(spi_ip.register_map)                   # 應列出 slv_reg0~5


False
['axi_rhd_spi_master_0', 'zynq_ultra_ps_e_0']
False


AttributeError: register_map only available if the .hwh is provided

In [50]:
# RHD SPI IP 的基地址 
RHD_SPI_BASE = 0xA0000000
ADDRESS_RANGE = 0x10000

# 暫存器偏移
CONTROL_REG = 0x00  # bit[0]=start_pulse, bit[1]=stop_pulse, bit[7:4]=phase_select
PHA_SEL_REG = 0x04  # [3:0]
TX_DATA_REG = 0x08  # 16-bit data to transmit
RX_DATA_REG = 0x0C  # 16-bit received data
STATUS_REG  = 0x10  # bit[0]=busy, bit[1]=finish
# 初始化 MMIO
mmio = MMIO(RHD_SPI_BASE, ADDRESS_RANGE)

# 讀取狀態
status = mmio.read(STATUS_REG)
print(f"SPI Status: 0x{status:08x}")
print(f"Busy: {status & 0x1}")
print(f"Finish: {(status >> 1) & 0x1}")

SPI Status: 0x00000001
Busy: 1
Finish: 0


In [51]:
from pynq import MMIO
BASE   = 0xA0000000
RANGE  = 0x10000
USER_REG = 0x14                 # slv_reg5

mmio = MMIO(BASE, RANGE)

patterns = [0xCAFEBABE, 0x0BAD_F00D, 0x55AA55AA]

for p in patterns:
    mmio.write(USER_REG, p)
    r = mmio.read(USER_REG)
    print(f"Write 0x{p:08X}  → Read 0x{r:08X} "
          f"{'OK' if r==p else 'FAIL'}")



Write 0xCAFEBABE  → Read 0xCAFEBABE OK
Write 0x0BADF00D  → Read 0x0BADF00D OK
Write 0x55AA55AA  → Read 0x55AA55AA OK


In [52]:
BUSY_MASK    = 0x1      # STATUS bit0
FINISH_MASK  = 0x2      # STATUS bit1

for _ in range(20):
    stat = mmio.read(STATUS)
    print(f"{stat:08X}  busy={stat&BUSY_MASK}  finish={(stat>>1)&1}")
    time.sleep(0.1)     # 每 100 ms 印一次


00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0
00000001  busy=1  finish=0


In [21]:
import time
def poll_one_sample(timeout=1.0):
    t0 = time.time()
    while (mmio.read(STATUS) & FINISH_MASK) == 0:
        if time.time()-t0 > timeout:
            raise TimeoutError("No FINISH within 1 s")
    data = mmio.read(RX_DATA) & 0xFFFF
    # 等 FINISH 回 0，避免重覆
    while mmio.read(STATUS) & FINISH_MASK:
        pass
    return data

# 抓 8 筆試試
print([hex(poll_one_sample()) for _ in range(8)])


TimeoutError: No FINISH within 1 s

In [87]:
# AXI 地址
CTRL    = 0x00
STATUS  = 0x10
RX_DATA = 0x0C
PH_SEL  = 0x04

# 1️⃣ 送 start_pulse：bit0 = 1
mmio.write(PH_SEL, 0x1)
mmio.write(CTRL, 0x1)

# 2️⃣ 等 BUSY 變 0（或 FINISH 變 1）
while mmio.read(STATUS) & 0x1:        # BUSY 還在
    pass

# 3️⃣ 此時 FINISH = 1，讀 RX_DATA
data = mmio.read(RX_DATA) & 0xFFFF
print(f"RX = 0x{data:04X}")



KeyboardInterrupt



In [87]:
def read_auto(n):
    vals = []
    for _ in range(n):
        mmio.write(CTRL, 0x1)                  # start
        while mmio.read(STATUS) & 0x1:         # busy
            pass
        vals.append(mmio.read(RX_DATA) & 0xFFFF)
    return vals

print(read_auto(40))


KeyboardInterrupt: 

In [86]:
PHASE_SEL = 0x04   # 你的 phase_select 暫存器
CTRL      = 0x00
STATUS    = 0x10
RX_DATA   = 0x0C

def read_once():
    mmio.write(CTRL, 0x1)          # start_pulse
    while mmio.read(STATUS) & 1:   # 等 busy=0
        pass
    return mmio.read(RX_DATA) & 0xFFFF

results = {}
for ph in range(16):               # 0‥15 全掃
    mmio.write(PHASE_SEL, ph)
    results[ph] = read_once()

for ph,val in results.items():
    print(f"phase {ph:2d}: 0x{val:04X}")


KeyboardInterrupt: 

In [90]:
data = mmio.read(RX_DATA) & 0xFFFF
print(f"RX = 0x{data:04X}")

RX = 0x0000
